In [ ]:
import os
import wget
import tarfile

# Gathering D1.002 source data
# Modify PATH at will
PATH_TO_InDATA = "../data/"
link = "https://zenodo.org/records/13993758/files/D1.tar.gz?download=1"

# Create path
os.makedirs(PATH_TO_InDATA, exist_ok=True)

In [ ]:
# Download
def download_data(PATH_TO_InDATA):
    os.chdir(PATH_TO_InDATA)
    wget.download(link, out=PATH_TO_InDATA )

download_data(PATH_TO_InDATA)

In [ ]:
def decompress_data(PATH_TO_FILE, PATH_TO_InDATA):
    os.makedirs(PATH_TO_InDATA, exist_ok=True)
    with tarfile.open(PATH_TO_FILE, "r:gz") as tar:
        tar.extractall(path=PATH_TO_InDATA)

# Modify PATHS at will
PATH_TO_FILE = os.path.join(PATH_TO_InDATA, "D1.tar.gz")
decompress_data(PATH_TO_FILE, PATH_TO_InDATA)

In [ ]:
# Specify the location of the CC config file. 
# os.environ['CC_CONFIG'] = '/path/to/your_cc_config.json'  #  e.g. chemicalchecker/setup/cc_config.json
os.environ['CC_CONFIG'] = '/aloy/home/acomajuncosa/cc_config.json'

from chemicalchecker import ChemicalChecker
ChemicalChecker.set_verbosity('DEBUG') # CRITICAL, ERROR, WARN, INFO or DEBUG
from chemicalchecker.core import DataSignature
import numpy as np
import pandas as pd
import json
%matplotlib inline

In [ ]:
local_cc_dir = '../local_CC_D1'
PATH_TO_DATA = "/aloy/home/acomajuncosa/CC_DATA/DATA/"  # See Download_Data.ipynb // Procedure step 3
# PATH_TO_DATA = "/aloy/web_checker/package_cc/2021_07/sign_model_links/"
cc_local = ChemicalChecker(local_cc_dir, dbconnect=False, custom_data_path=PATH_TO_DATA)

## LOAD INPUT DATA -- NEW D1.002 SPACE ##

In [ ]:
inputFile = "../data/D1/chdir_active_10.h5"
M = np.array(DataSignature(inputFile))
print(min(M.flatten()), max(M.flatten()), M.shape)

## sign0 ##

In [ ]:
# Dataset Name
dataset = 'D1.002'

# Instantiation of sign0 data structures for the new space: full and reference
sign0 = cc_local.signature(dataset, 'sign0')

# Cleaning both full and reference datasets. This is crucial!
sign0.clear_all()

# Fit sign0
sign0.fit(data_file=inputFile, sanitizer_kwargs={"max_features": 13000})

In [ ]:
sign0.shape

In [ ]:
# Instantiation of sign0
sign0 = cc_local.signature(dataset, 'sign0')

# Instantiation of diag0 (diagnosis plots)
diag0 = sign0.diagnosis()

# Plot small diagnosis plots
diag0.canvas(size='small', savefig=True, savefig_kwargs={'dpi': 300})

In [ ]:
np.min(np.array(sign0).flatten()), np.max(np.array(sign0).flatten())

## sign1 ##

In [ ]:
# Dataset Name
dataset = 'D1.002'

# Instantiation of sign0
sign0 = cc_local.signature(dataset, 'sign0')

# Instantiation of sign1
sign1 = cc_local.signature(dataset, 'sign1')

# Cleaning both full and reference datasets. This is crucial!
sign1.clear_all()

# Fitting sign1
sign1.fit(sign0, scale_kwargs=dict(max_keys=13000), pca_kwargs=dict(max_keys=13000))

In [ ]:
sign1.shape

In [ ]:
# Instantiation of sign1
sign1 = cc_local.signature(dataset, 'sign1')

# Instantiation of diag1 (diagnosis plots)
diag1 = sign1.diagnosis()

# Plot medium & small diagnosis plots
diag1.canvas(size='small', savefig=True, savefig_kwargs={'dpi': 300})

In [ ]:
# Instantiation of sign1
sign1 = cc_local.signature(dataset, 'sign1')

# Instantiation of neig1
neig1 = cc_local.get_signature("neig1", "full", dataset)  # It will take the reference anyway...

# Cleaning both full and reference. This is crucial!
neig1.clear_all()

# Fitting neig1
neig1.fit(sign1)

In [ ]:
neig1.shape

In [ ]:
np.min(np.array(sign1).flatten()), np.max(np.array(sign1).flatten())

## sign2 ##

In [ ]:
# Dataset Name
dataset = 'D1.002'

# Get sign1
sign1 = cc_local.get_signature('sign1', 'full', dataset)

# Get neig1
neig1 = cc_local.get_signature('neig1', 'full', dataset)  # By default, all vs ref

# Instantiation of sign2
sign2 = cc_local.signature(dataset, 'sign2')

# Cleaning both full and reference datasets. This is crucial!
sign2.clear_all()

# Fit sign2 given sign1 & neig1
sign2.fit(sign1, neig1, oos_predictor=False)

In [ ]:
sign2.shape

In [ ]:
# Instantiation of sign2
sign2 = cc_local.signature(dataset, 'sign2')

# Instantiation of diag2 (diagnosis plots)
diag2 = sign2.diagnosis()

# Plot medium & small diagnosis plots
diag2.canvas(size='small', savefig=True, savefig_kwargs={'dpi': 300})

In [ ]:
np.min(np.array(sign2).flatten()), np.max(np.array(sign2).flatten())

## sign3 ##

In [ ]:
# Dataset Name
dataset = 'D1.002'

# Get CC universe
cc_universe = []
for dat in cc_local.datasets:
    if dat != dataset and dat.endswith('001') and "D1" not in dat:
        cc_universe.extend(cc_local.get_signature('sign2', 'full', dat).keys)
cc_universe = set(cc_universe)

# Get sign2
sign2 = cc_local.signature(dataset, 'sign2')

# Get D1.002 molecules
d1_molecules = set(sign2.keys)

print("Number of molecules in the CC universe: " + str(len(cc_universe)))
print("Number of molecules in D1.002 sign2: " + str(len(d1_molecules)))
print("Intersection CC & D1.002: " + str(len(cc_universe.intersection(d1_molecules))))

In [ ]:
# Dataset Name
dataset = 'D1.002'

# Instantiation of sign3
sign3 = cc_local.signature(dataset, 'sign3')
sign3.clear_all()

# Create a list of sign2 to feed sign3 -- using the 24 CC spaces (without D1.001) and D1.002
sign2_list = list()

# For each CC space
for ds in cc_local.coordinates:
    ds += '.001'
    if ds == 'D1.001':
        ds = 'D1.002'
    sign2_list.append(cc_local.get_signature('sign2', 'full', ds))

# In total, we now have 25 spaces
print(len(sign2_list))

# Get D1.002 sign1
sign1_self = cc_local.signature(dataset, 'sign1')

# Get D1.002 sign2
sign2_self = cc_local.signature(dataset, 'sign2')

In [ ]:
# Fit sign3 

# Option 1: in the cluster
# kwargs = {"cpu": 5, "mem_by_core": 12, "wait": False}  # GPU SBNB-IRB CONFIGURATION
# kwargs = {"cpu": 32, "mem_by_core": 1, "wait": False}  # CPU SBNB-IRB CONFIGURATION
# sign3.fit_hpc(sign2_list, sign2_self, sign1_self, sign2_universe=None, complete_universe="full", sign2_coverage=None, hpc_args=kwargs)

# Option 2: locally  // CAUTION: COMPUTATIONALLY DEMANDING STEP
sign3.fit(sign2_list, sign2_self, sign1_self, sign2_universe=None, complete_universe=False, sign2_coverage=None)  # Consider not completing the universe

In [ ]:
dataset = 'D1.002'

# Instantiation of sign3
sign3 = cc_local.signature(dataset, 'sign3')
sign3 = np.array(sign3)

print(sign3.shape, np.min(sign3), np.max(sign3))

In [ ]:
# Instantiation of diag3 (diagnosis plots)
sign3 = cc_local.signature(dataset, 'sign3')
diag3 = sign3.diagnosis(ref_cctype='sign3')

# Plot medium & small diagnosis plots
diag3.canvas(size='medium', savefig=True, savefig_kwargs={'dpi': 300})
diag3.canvas(size='small', savefig=True, savefig_kwargs={'dpi': 300})